# ISIMIP — bias-adjusted, impact-ready climate forcing (quickstart)

[`earthlens.isimip`](../../reference/isimip/introduction.md) fetches the **ISIMIP**
repository of **bias-adjusted, impact-model-ready climate forcing** — CMIP6 GCMs already
bias-corrected against the W5E5 observational dataset, exactly as the hydrology / flood
impact-modelling community consumes them.

**`isimip` vs `cmip6`.** The `cmip6` backend gives you the *raw* CMIP6 archive on each
model's native grid — you would have to bias-adjust and regrid it yourself before driving an
impact model. `isimip` gives you that step already done. Reach for `isimip` when you want
impact-ready forcing out of the box.

**The cutout is mandatory.** A single ISIMIP global-daily granule is ~1–2 GB (a whole dataset
is ~18 GB), so `earthlens.isimip` never pulls one whole for a regional request: it submits a
**server-side cutout job** and downloads only the requested box — a 2° × 2° European cutout of
one decade granule is ~4 MB instead of ~1.2 GB.

By the end you will build a facet request, run a live cutout, and read + plot the returned
NetCDF with pyramids.

## Setup

`EarthLens` is the unified entry point; `earthlens.isimip.Catalog` is the config +
curated-facet-vocabulary catalog. `tempfile` / `pathlib` give us a scratch directory for the
cut NetCDF. The backend needs the `isimip` extra (`pip install "earthlens[isimip]"`), which
adds `isimip-client`.

In [ ]:
import tempfile
from pathlib import Path

from earthlens.core import EarthLens
from earthlens.isimip import Catalog

## Explore the facet vocabulary (no network)

A request pins ISIMIP facets — the simulation round, the GCM (`climate_forcing`), the
scenario, and the variable. The bundled catalog carries the curated vocabulary and the API +
cutout endpoints; facets are validated against it, so a typo raises a clear did-you-mean error
before any network call.

In [ ]:
cat = Catalog()
print(
    'variable :', cat.get_dataset('pr').long_name, '(', cat.get_dataset('pr').units, ')'
)
print('forcing  :', 'gfdl-esm4 ->', cat.get_forcing('gfdl-esm4').description)
print('scenario :', 'ssp585 ->', cat.get_scenario('ssp585').description)
print('round    :', 'ISIMIP3b ->', cat.get_round('ISIMIP3b').default_license)
print('API      :', cat.data_url)

## Build and run the request (live cutout)

The request is an ISIMIP **facet set** plus a date window and a bounding box. We cut to a small
European box and a one-year window; `download()` submits the cutout job (submit → poll →
download), and returns the `list[Path]` of the cut NetCDF granule(s).

| Argument | Meaning | Value here |
|----------|---------|------------|
| `dataset` | simulation round | `ISIMIP3b` |
| `gcm` | climate forcing (GCM) | `gfdl-esm4` |
| `scenario` | emissions scenario | `ssp585` |
| `variables` | climate variable(s) | `['pr']` (precipitation) |
| `lat_lim` / `lon_lim` | cutout bbox | `[51, 53]` / `[6, 8]` |

In [ ]:
out_dir = Path(tempfile.mkdtemp(prefix='isimip_'))
paths = EarthLens(
    'isimip',
    dataset='ISIMIP3b',
    gcm='gfdl-esm4',
    scenario='ssp585',
    variables=['pr'],
    start='2030-01-01',
    end='2030-12-31',
    lat_lim=[51.0, 53.0],
    lon_lim=[6.0, 8.0],
    path=str(out_dir),
).download(progress_bar=False)
paths

The returned path is the granule **cut to our bbox** — a few MB, not the ~1.2 GB global
source file. Reading and regridding it is [pyramids](https://github.com/serapeum-org/pyramids)'
job; `earthlens` never imports `xarray` / `netCDF4`.

## Inspect the written NetCDF with pyramids

`pyramids.netcdf.NetCDF.read_file` opens the granule as a `Container`. The bias-adjusted
precipitation variable `pr` carries one band per daily timestep of the cut decade granule.

In [ ]:
from pyramids.netcdf import NetCDF

container = NetCDF.read_file(paths[0])
print('variables :', container.variable_names)
pr = container.get_variable('pr')
print('daily bands:', pr.band_count)

## Plot the bias-adjusted precipitation field

Each band is a georeferenced grid over our cutout box. We plot the field with pyramids (via
cleopatra) — a small window of bias-adjusted daily precipitation, ready to drive an impact
model.

In [ ]:
glyph = pr.plot(title='ISIMIP3b GFDL-ESM4 ssp585 — bias-adjusted daily precipitation')

## Licence and attribution

ISIMIP licences are **per dataset** and read live from the repository. Most ISIMIP3b InputData
(the W5E5-bias-adjusted forcing) is **`CC0 1.0`** (public domain); some OutputData / sectoral
inputs carry their own terms and are flagged `restricted` (the backend warns for those). Always
cite ISIMIP and the underlying GCM.

In [ ]:
backend = EarthLens(
    'isimip',
    dataset='ISIMIP3b',
    gcm='gfdl-esm4',
    scenario='ssp585',
    variables=['pr'],
    start='2030-01-01',
    end='2030-12-31',
    lat_lim=[51.0, 53.0],
    lon_lim=[6.0, 8.0],
    path=str(out_dir),
).datasource
print(backend.terms_note())

## Takeaway

- `EarthLens('isimip', ...)` fetches **bias-adjusted, impact-ready** climate forcing — the
  already-corrected counterpart to the raw `cmip6` archive.
- The request is a **facet set** (round / GCM / scenario / variable), validated against the
  catalog before any network call.
- The **server-side cutout is mandatory**: you get the bbox cut (a few MB), never the ~18 GB
  global dataset. Pass `whole_globe=True` only if you really want the raw global granules.
- `download()` returns raw NetCDF paths; read / regrid / reduce them with **pyramids**.

See the [usage guide](../../reference/isimip/usage.md) for multi-variable requests and the
whole-globe opt-in.